# 프로젝트: KoChatGPT 업그레이드 하기 (LLM Trend Note 2)

KoChatGPT는 SFT → Reward Model → PPO 로 이어지는 RLHF 파이프라인을 KoGPT-2 위에 얹은 한국어 ChatGPT 복제 실습이다.  
이번 프로젝트는 그 baseline을 그대로 쓰는 게 아니라, 제시된 업그레이드 전략 중 하나를 골라 모델을 개선하고, 그 개선이 실제로 출력을 어떻게 바꾸는지를 정량·정성적으로 비교하는 게 목표다.

내 트랙이 모델 내부 최적화 쪽은 아니라서, 이번 프로젝트는 **점수를 극한까지 끌어올리는 것보다 "데이터 정제와 디코딩 전략이 모델 동작을 어떻게 바꾸는지"를 관찰하는 데 초점**을 맞췄다. 그래서 하이퍼파라미터를 붙잡고 튜닝하기보다, 각 단계(기존 KoGPT2 → SFT → RLHF)에서 출력이 어떻게 달라지는지를 눈으로 보고 지표로 확인하는 흐름으로 짰다.

## Step 1. 프로젝트 개요와 전략 선택

### 업그레이드 전략 (지시사항)
프로젝트에서 제시된 개선 전략은 크게 세 가지였다.
1. **기존 데이터셋 추가 정제** — 말뭉치를 EDA해서 도메인/길이분포/문장 완성도를 분석하고, 정제해서 재학습. (정제 후 크기가 줄지 않도록 augmentation으로 유지 내지 증량, BLEU/ROUGE 등 정량 비교)
2. **새로운 데이터셋 추가** — 웹크롤링 등으로 ranking dataset을 새로 구축해 추가.
3. **foundation model 교체** — KoGPT-2 대신 더 큰 모델(예: skt/ko-gpt-trinity-1.2B)로 교체.

### 내가 고른 전략: 1번 (데이터 추가 정제) + 디코딩 기법 실험
세 개 중 1번을 골랐다. 이유는 세 가지다.
- 2번(웹크롤링 새 데이터 구축)은 시간이 많이 들고, 3번(1.2B 모델 교체)은 코랩 GPU에서 OOM 씨름이 커서, 정해진 시간 안에 "개선 → 관찰" 사이클을 여러 번 돌리기엔 1번이 제일 낫다고 판단했다.
- 무엇보다 EDA를 해보니 기존 SFT 데이터에 **정제할 여지가 뚜렷하게 많았다**(아래 Step 3에서 확인). 데이터 품질이 생성 성능에 큰 영향을 준다는 걸 직접 확인하기 좋은 소재였다.
- 루브릭 1번이 "정제 + generation 기법(beam search, top-k 등) 실험으로 성능 향상"이라 1번 전략과 정확히 맞물린다.

### 루브릭 대조 (구현 계획)
| 평가기준 | 이 노트북에서 어디서 충족하나 |
|---|---|
| 1. 데이터 정제 / 새 데이터 / foundation model 교체 중 하나로 정량적 성능 향상 (Beam search, Top-k 등 실험) | Step 3~4 정제·증강 + Step 11 디코딩 기법 실험(greedy/beam/top-k/top-p) + BLEU/ROUGE 비교 |
| 2. SFT 모델과 RM(=RLHF 적용) 모델 결과 분석 | Step 8~10: RM 학습 → PPO 학습 → SFT vs PPO 출력 비교(정성) + RM 점수 비교(정량) |
| 3. 기존 KoGPT2와 SFT 적용 모델 결과 분석 | Step 5 기존 KoGPT2 baseline → Step 7 SFT 결과와 정성/정량 비교 |

전체 흐름: **환경설정 → EDA → 정제·증강 → (기존 KoGPT2) → SFT → 비교 → RM → PPO → 비교 → 디코딩 실험 → 종합**

## Step 2. 환경 설정 (Colab)

KoChatGPT 레포를 클론하고 데이터와 `chatgpt` 패키지를 설치한다.  
원본 노드는 colossalai(멀티 GPU 분산학습용)를 쓰는데, 나는 코랩 single GPU만 쓸 거고 옛날 colossalai+torch 조합이 최신 코랩에서 설치가 잘 안 됐다. 그래서 `chatgpt` 패키지에서 **naive(단일 GPU) 전략만 남기고 colossalai 의존을 걷어낸 뒤** 설치한다. `strategies/__init__.py`가 colossalai 전략을 import하고 있어서 이 파일을 교체하는데, `save_checkpoint` 콜백이 `ColossalAIStrategy`라는 이름을 `isinstance` 체크용으로 import하고 있어서, 이름 자체는 colossalai 없는 **스텁 클래스**로 남겨둔다. 이렇게 하면 naive 전략은 그 스텁의 인스턴스가 아니라 항상 False가 나와서 원래 동작과 똑같고, colossalai 없이도 패키지가 import된다.

> transformers/accelerate는 처음에 원본 노드처럼 옛날 버전으로 내리려 했는데, 최신 코랩에서는 오히려 그게 문제였다. 옛날 accelerate엔 최신 transformers가 찾는 함수가 없어서 `Trainer` import 자체가 깨졌다. 그래서 버전을 내리는 대신 **transformers와 accelerate를 호환되는 최신 짝으로 함께 올려서** 맞췄다. 이 설치 셀을 처음 실행한 뒤에는 `런타임 → 세션 다시 시작`을 한 번 해줘야 이미 로드돼 있던 옛 버전 대신 새 버전이 반영된다. 재시작 후 Step 2를 위에서부터 다시 실행하면 된다.

In [1]:
# 레포 클론 + 데이터 준비
import os
if not os.path.exists('KoChatGPT'):
    !git clone https://github.com/airobotlab/KoChatGPT
    !cp -r KoChatGPT/data_kochatgpt .
    !cp -r KoChatGPT/img . 2>/dev/null || true

# colossalai 의존 걷어내기: strategies/__init__.py를 naive 전략 중심으로 교체한다.
# 단, save_checkpoint 콜백이 ColossalAIStrategy 이름을 isinstance 체크용으로 import하므로,
# colossalai 없이 이름만 살려두는 스텁 클래스로 남긴다. (naive는 이 클래스 인스턴스가 아니라 항상 False)
init_path = 'KoChatGPT/colossalai_ChatGPT_230319/chatgpt/trainer/strategies/__init__.py'
with open(init_path, 'w') as f:
    f.write("from .base import Strategy\n")
    f.write("from .naive import NaiveStrategy\n\n")
    f.write("class ColossalAIStrategy:  # colossalai 없이 이름만 살려두는 스텁\n")
    f.write("    pass\n\n")
    f.write("__all__ = ['Strategy', 'NaiveStrategy', 'ColossalAIStrategy']\n")
print('patched:', init_path)

Cloning into 'KoChatGPT'...
remote: Enumerating objects: 304, done.
remote: Total 304 (delta 0), reused 0 (delta 0), pack-reused 304 (from 1)
Receiving objects: 100% (304/304), 57.72 MiB | 19.45 MiB/s, done.
Resolving deltas: 100% (123/123), done.
patched: KoChatGPT/colossalai_ChatGPT_230319/chatgpt/trainer/strategies/__init__.py


In [2]:
# chatgpt 패키지 설치 (colossalai 없이) + 필요한 라이브러리
# --no-deps 로 설치해서 requirements.txt에 박힌 colossalai가 안 딸려오게 한다.
%cd KoChatGPT/colossalai_ChatGPT_230319/
!pip install . --no-deps -q
%cd /content

# transformers/accelerate는 옛날 버전으로 내리면 서로 짝이 안 맞아 Trainer import가 깨진다
# (accelerate에 있어야 할 함수를 transformers가 못 찾는 식). 그래서 버전을 내리지 않고
# 코랩에서 서로 호환되는 최신 짝으로 올려서 맞춘다.
!pip install -q -U transformers accelerate
!pip install -q datasets loralib rouge-score sacrebleu
print('done  (이 셀을 처음 돌린 뒤에는 런타임을 한 번 재시작해야 새 버전이 반영된다)')

/content/KoChatGPT/colossalai_ChatGPT_230319
  Preparing metadata (setup.py) ... done
/content
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 84.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatgpt 0.1.0 requires colossalai>=0.2.4, which is not installed.
chatgpt 0.1.0 requires loralib, which is not installed.
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 5.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
chatgpt 0.1.0 requires colossalai>=0.2.4, which is not installed.
done  (이 셀을 처음 돌린 뒤에는 런타임을 한 번 재시작해야 새 버전이 반영된다)


In [3]:
import torch
print('torch', torch.__version__, '| cuda available:', torch.cuda.is_available())
# naive 전략이 colossalai 없이 import되는지 확인
from chatgpt.trainer.strategies import NaiveStrategy
print('NaiveStrategy import OK')

torch 2.11.0+cu128 | cuda available: True
NaiveStrategy import OK


## Step 3. 데이터 EDA

정제를 하려면 먼저 **무엇을 정제할지**를 데이터에서 근거로 찾아야 한다. 세 말뭉치(SFT/RM/PPO)를 열어서 개수, 길이 분포, 그리고 품질 신호(영어 범벅 응답, 너무 짧은 응답, 이상한 따옴표, 미완결 문장, 중복 프롬프트)를 확인한다.

In [4]:
import json, re
import statistics as st

def load_json(path):
    with open(path, 'r', encoding='utf-8-sig') as f:
        return json.load(f)

sft  = load_json('data_kochatgpt/kochatgpt_1_SFT.jsonl')
rm   = load_json('data_kochatgpt/kochatgpt_2_RM.jsonl')
ppo  = load_json('data_kochatgpt/kochatgpt_3_PPO.jsonl')

print('SFT :', len(sft), '| keys:', list(sft[0].keys()))
print('RM  :', len(rm),  '| keys:', list(rm[0].keys()))
print('PPO :', len(ppo), '| keys:', list(ppo[0].keys()))

SFT : 12000 | keys: ['prompt', 'completion', 'tokens']
RM  : 10220 | keys: ['prompt', 'completion_0', 'completion_1', 'completion_2', 'ranking']
PPO : 12000 | keys: ['prompt']


In [5]:
# SFT completion 길이 분포
lens = [len(d['completion']) for d in sft]
print('completion 글자수 | min %d / median %d / mean %.0f / max %d'
      % (min(lens), st.median(lens), st.mean(lens), max(lens)))

# 품질 신호별로 개수를 세본다 (여기서 나온 숫자가 곧 정제 규칙의 근거가 된다)
def english_heavy(t):
    lat = len(re.findall(r'[A-Za-z]', t)); ko = len(re.findall(r'[가-힣]', t))
    return lat > max(10, ko)          # 한글보다 라틴문자가 많으면 영어 범벅으로 본다

QUOTES = "\'\""
n_eng      = sum(english_heavy(d['completion']) for d in sft)
n_short    = sum(len(d['completion'].strip()) < 10 for d in sft)
n_quote    = sum(d['completion'].strip()[:1] in QUOTES for d in sft)
n_literal  = sum('\\n' in d['completion'] for d in sft)
n_dup      = len(sft) - len(set(d['prompt'] for d in sft))

print('영어 범벅 응답      :', n_eng)
print('너무 짧은 응답(<10) :', n_short)
print('따옴표로 시작       :', n_quote, '(전체의 %.1f%%)' % (100*n_quote/len(sft)))
print('literal \\n 포함     :', n_literal)
print('중복 프롬프트       :', n_dup)

completion 글자수 | min 4 / median 118 / mean 144 / max 1553
영어 범벅 응답      : 534
너무 짧은 응답(<10) : 115
따옴표로 시작       : 11975 (전체의 99.8%)
literal \n 포함     : 1238
중복 프롬프트       : 54


EDA 결과를 보면 정제 포인트가 분명하다. 특히 **거의 모든 completion(99% 이상)이 stray 따옴표로 시작**하는데, 이건 원본 데이터를 JSON으로 만들 때 문자열이 한 번 더 감싸진 흔적으로 보인다. 그리고 응답 안에 진짜 줄바꿈 대신 `\n` 이라는 글자 두 개가 그대로 박혀 있는 경우, 한글은 하나도 없이 영어/불어/스페인어로만 답한 경우도 꽤 있다. 이런 것들이 학습 노이즈가 되니 정제 대상으로 잡는다.

## Step 4. 데이터 정제 + 증강

EDA에서 찾은 문제들을 정제 규칙으로 옮긴다.
- **정제(clean_completion):** 앞뒤 stray 따옴표 제거, literal `\n`/`\t`를 진짜 공백·줄바꿈으로 복원, 공백 정리.
- **필터(is_bad):** 너무 짧거나(<10자), 한글이 없거나, 영어 범벅인 응답은 통째로 제외. 프롬프트 중복도 제거.

지시사항에 **"정제 후 데이터셋 크기가 줄어들지 않도록 augmentation으로 유지 내지 증량"** 하라고 했으니, 필터로 줄어든 만큼 다시 채운다. 증강은 completion(정답)을 건드리면 의미가 망가질 수 있어서, **정답은 그대로 두고 instruction(프롬프트)만 여러 표현으로 감싸는 방식**을 썼다. 같은 답을 다양한 질문 형태로 보여주면 instruction 다양성이 올라가는 효과가 있고, 정답 품질은 안 깨진다. (증강 자체를 정교하게 튜닝하는 건 이번 스코프가 아니라서 단순하고 안전한 방법으로만 했다.)

In [6]:
QUOTES = "\'\""

def clean_completion(t):
    t = t.strip()
    t = t.strip(QUOTES)                                  # 앞뒤 stray 따옴표 제거
    t = t.replace('\\n', '\n').replace('\\t', ' ')      # literal 이스케이프 복원
    t = re.sub(r'[ \t]+', ' ', t)                        # 연속 공백 정리
    t = re.sub(r'\n{3,}', '\n\n', t)                    # 과한 줄바꿈 정리
    return t.strip()

def is_bad(prompt, comp):
    c = comp.strip()
    if len(c) < 10: return True                          # 너무 짧음
    ko  = len(re.findall(r'[가-힣]', c))
    lat = len(re.findall(r'[A-Za-z]', c))
    if ko == 0: return True                              # 한글 아예 없음
    if lat > max(10, ko): return True                    # 영어 범벅
    return False

seen = set()
cleaned = []
for d in sft:
    comp = clean_completion(d['completion'])
    if d['prompt'] in seen:      # 프롬프트 중복 제거
        continue
    if is_bad(d['prompt'], comp):
        continue
    seen.add(d['prompt'])
    cleaned.append({'prompt': d['prompt'], 'completion': comp})

print('정제 전:', len(sft))
print('정제 후:', len(cleaned), '(제거 %d개)' % (len(sft) - len(cleaned)))
print('\n[정제 예시]')
print('before:', repr(sft[0]['completion'][:70]))
print('after :', repr(cleaned[0]['completion'][:70]))

정제 전: 12000
정제 후: 11230 (제거 770개)

[정제 예시]
before: "'저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고"
after : '저는 인공지능 챗봇이며, 직접적으로 식품에 관한 정보를 가지고 있지 않습니다. 하지만 일반적으로 불고기용 고기는 한우, 쇠고기'


In [7]:
# 증강: 정답은 그대로, instruction만 다양한 표현으로 감싸서 크기를 원본 이상으로 복원한다.
import random
random.seed(42)

templates = [
    "{p}",                          # 원형
    "다음 질문에 답해줘: {p}",
    "{p} 알려줘.",
    "질문: {p}",
]

target_size = len(sft)              # 원본(12000) 이상으로 맞춘다
augmented = list(cleaned)           # 정제본을 먼저 담고
need = max(0, target_size - len(cleaned))

pool = cleaned.copy()
random.shuffle(pool)
i = 0
while len(augmented) < target_size and pool:
    base = pool[i % len(pool)]
    tmpl = random.choice(templates[1:])            # 원형 말고 변형 템플릿
    new_prompt = tmpl.format(p=base['prompt'])
    augmented.append({'prompt': new_prompt, 'completion': base['completion']})
    i += 1

print('증강 후 최종:', len(augmented), '(원본 %d 대비 유지/증량 확인)' % len(sft))

# train / eval 분리 — eval은 정량평가(BLEU/ROUGE)용 held-out
random.shuffle(augmented)
eval_size = 200
eval_set  = augmented[:eval_size]
train_set = augmented[eval_size:]
print('train:', len(train_set), '| eval(held-out):', len(eval_set))

# 정제된 학습셋 저장 (SFT_dataset이 json.load로 읽을 수 있게 JSON 배열로 저장)
with open('data_kochatgpt/kochatgpt_1_SFT_clean.jsonl', 'w', encoding='utf-8') as f:
    json.dump(train_set, f, ensure_ascii=False)
with open('data_kochatgpt/kochatgpt_1_SFT_eval.jsonl', 'w', encoding='utf-8') as f:
    json.dump(eval_set, f, ensure_ascii=False)
print('saved cleaned train/eval')

증강 후 최종: 12000 (원본 12000 대비 유지/증량 확인)
train: 11800 | eval(held-out): 200
saved cleaned train/eval


## Step 5. 베이스라인 — 기존 KoGPT2 인퍼런스

SFT를 하기 전에, 순수 KoGPT-2가 우리 프롬프트 형식에 어떻게 답하는지를 먼저 찍어둔다. 이게 루브릭 3번(기존 KoGPT2 vs SFT)의 비교 기준선이 된다. KoGPT-2는 그냥 다음 단어를 이어붙이는 언어모델이라, "지시를 따르는" 훈련을 안 받은 상태에서 어떻게 헤매는지를 볼 수 있다.

In [8]:
import torch
from transformers import PreTrainedTokenizerFast, GPT2LMHeadModel

# 프롬프트 템플릿 (SFT 학습 때와 동일하게 맞춰야 비교가 공정하다)
PROMPT_DICT = {
    "prompt_no_input": (
        "아래는 작업을 설명하는 명령어입니다.\n"
        "명령어에 따른 요청을 적절히 완료하는 응답을 작성하세요.\n\n"
        "### Instruction(명령어):\n{prompt}\n\n### Response(응답):"
    )
}

# 평가용 고정 프롬프트 (정성 비교용, 단계마다 같은 걸 쓴다)
eval_prompts_raw = [
    '불고기용 고기 한우에요?',
    '리처드 닉슨이 43대 부통령직을 수행한 년도는?',
    '시카고 오헤어 국제공항은 어디에 있어',
    '오늘 미세먼지 어때?'
]
eval_prompts = [PROMPT_DICT['prompt_no_input'].format_map({'prompt': p}) for p in eval_prompts_raw]

base_tok = PreTrainedTokenizerFast.from_pretrained(
    "skt/kogpt2-base-v2",
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>')
base_model = GPT2LMHeadModel.from_pretrained('skt/kogpt2-base-v2').to(
    'cuda' if torch.cuda.is_available() else 'cpu')

def base_generate(text):
    ids = base_tok.encode(text, return_tensors='pt').to(base_model.device)
    out = base_model.generate(ids, max_length=200, repetition_penalty=2.0,
                              pad_token_id=base_tok.pad_token_id,
                              eos_token_id=base_tok.eos_token_id, use_cache=True)
    return base_tok.decode(out[0], skip_special_tokens=True)

print('=== 기존 KoGPT2 (SFT 전) 응답 ===')
base_outputs = {}
for raw, full in zip(eval_prompts_raw, eval_prompts):
    gen = base_generate(full)
    ans = gen.split('### Response(응답):')[-1].strip()
    base_outputs[raw] = ans
    print('\nQ:', raw)
    print('A:', ans[:200])

tokenizer.json:   0%|          | 0.00/2.83M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.00k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  513MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

model.safetensors: reconstructing file:   0%|          |  0.00B /  513MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


=== 기존 KoGPT2 (SFT 전) 응답 ===

Q: 불고기용 고기 한우에요?
A: food_delicious. 오늘은 제가 좋아하는 과자랑 함께 먹었어요.
요즘 날씨가 너무 추워서 그런지 더더욱 많이 먹고 싶었는데요.
오늘도 역시나 맛있는 과일들~!!
이렇게 맛있게 잘 먹는거 보니까 정말 기분이 좋네요
저는 이번에 처음 먹어보는 딸기잼과 딸기를 좋아하는데요,
딸기쨈에 들어있는 베리베리향이 참 매력적이에요.
그리고 달달한 맛이 일품이라서 

Q: 리처드 닉슨이 43대 부통령직을 수행한 년도는?
A: beatmania_mygame.com/views/2648297.
이번주 토요일 오후 4시 30분부터 5시까지 서울 강남구 논현동 강남역 10길에서 열리는 ``2012 대한민국 게임대상 시상식 및 시상식: GAMESHOWN 2016-19 (Games of the Year) - Korea Top 100, 한국게임산업진흥원 주최로 열린다.' 라는 주제로 진행되는

Q: 시카고 오헤어 국제공항은 어디에 있어
A: jmtbc_kim (오늘도 좋은 하루 되세요)
이제 곧 여름휴가 시작!
아침부터 저녁까지 시원하게 보내주신다면! 오늘만큼 더위를 식혀줄 시원한 음료 한잔으로 즐거운 시간을 만들어보자구요
 
저는 이번에 처음 먹어보는 맥심 모카맛이라서 너무 좋았어요.
맥심은 정말 맛있었답니다.
그리고 저는 그 맛을 잊지 못할 것 같아요.
그런데 제가 좋아하는 치즈맛이랑 잘 

Q: 오늘 미세먼지 어때?
A: selfie (반응:)
이제부터 시작!
미세 먼지가 많은 날엔, 마스크를 착용하고 외출해야 합니다.
마지막으로 중요한 것은 바로 '호흡기' 입니다.
평소보다 더 자주 숨을 쉬어야 하고, 특히나 호흡기가 약한 사람은 더욱 그렇습니다.
그렇다면 어떻게 해야 할까요?
우선 호흡이 잘 안 되는 날은 가급적 외출을 자제하고, 실내에서 휴식을 취하도록 하십시오.
그리


## Step 6. SFT 학습

정제한 데이터로 SFT를 돌린다. 핵심은 **정답(Response) 부분에 대해서만 loss를 계산**하는 것이다. 입력은 `프롬프트 + 정답` 전체를 넣지만, 프롬프트 부분의 label은 `-100`(IGNORE_INDEX)으로 채워서 loss에서 제외한다. 이렇게 해야 모델이 "질문을 외우는" 게 아니라 "질문에 대한 답을 생성하는" 걸 배운다.  
(예전에 이 `-100` 마스킹을 빠뜨리면 프롬프트까지 학습돼서 이상하게 답하는 걸 겪은 적이 있어서, 이번엔 source 길이만큼 정확히 마스킹되는지 확인하면서 짰다.)

토크나이저 로드 방식도 하나 짚고 간다. 처음엔 `AutoTokenizer`로 로드했는데, 학습이 멀쩡히 돌고 loss도 내려가는데 **생성 결과가 전부 `�`로 깨졌다**. 원인을 추적해보니 최신 transformers에서 AutoTokenizer가 kogpt2를 byte-level BPE인 `GPT2Tokenizer`로 잡아버려서, kogpt2의 문자 단위 vocab과 인코딩/디코딩이 어긋난 거였다. Step 5에서 기존 KoGPT2가 멀쩡히 한국어를 뱉었던 로드 방식(`PreTrainedTokenizerFast` + 스페셜 토큰 명시, SKT 공식 방식)과의 차이가 결정적 단서였다. 그래서 SFT/RM/PPO 세 단계 모두 이 방식으로 통일했다. 덤으로 pad 토큰이 `</s>`(eos 겸용)가 아니라 전용 `<pad>`가 되면서, 정답 끝의 eos가 pad로 오인되지 않아 라벨 처리도 더 깔끔해졌다.

In [9]:
import copy, logging
import transformers
from transformers import AutoModelForCausalLM, Trainer, TrainingArguments, PreTrainedTokenizerFast
from torch.utils.data import Dataset
from dataclasses import dataclass
from typing import Dict, Sequence

IGNORE_INDEX = -100

model_name = 'skt/kogpt2-base-v2'
model = AutoModelForCausalLM.from_pretrained(model_name)

# 토크나이저는 AutoTokenizer가 아니라 PreTrainedTokenizerFast로 고정한다.
# AutoTokenizer로 로드했더니 GPT2Tokenizer(byte-level)로 잡혀서 kogpt2 vocab과 안 맞아
# 생성 결과가 �로 깨졌다. Step 5에서 정상 동작을 확인한 로드 방식(SKT 공식 방식)으로 통일한다.
tokenizer = PreTrainedTokenizerFast.from_pretrained(
    model_name,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
    padding_side='right', model_max_length=512)
print('tokenizer:', type(tokenizer).__name__, '| vocab:', len(tokenizer), '| pad =', tokenizer.pad_token)

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

[transformers] GPT2LMHeadModel LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer: TokenizersBackend | vocab: 51200 | pad = <pad>


In [10]:
class SFT_dataset(Dataset):
    def __init__(self, data_path, tokenizer, verbose=False):
        with open(data_path, 'r', encoding='utf-8-sig') as f:
            list_data_dict = json.load(f)

        prompt_tmpl = PROMPT_DICT['prompt_no_input']
        # 입력(source) = 프롬프트 템플릿, 출력(target) = 정답 + eos
        sources = [prompt_tmpl.format_map({'prompt': ex['prompt']}) for ex in list_data_dict]
        targets = [f"{ex['completion']}{tokenizer.eos_token}" for ex in list_data_dict]

        examples = [s + t for s, t in zip(sources, targets)]
        sources_tok  = self._tok(sources, tokenizer)     # source만 (길이 측정용)
        examples_tok = self._tok(examples, tokenizer)     # source + target

        input_ids = examples_tok['input_ids']
        labels = copy.deepcopy(input_ids)
        # source 길이만큼은 -100으로 마스킹 → target(정답)에 대해서만 loss
        for label, src_len in zip(labels, sources_tok['input_ids_lens']):
            label[:src_len] = IGNORE_INDEX
        self.input_ids, self.labels = input_ids, labels
        if verbose:
            print('sample source:', sources[0][:60], '...')
            print('loaded:', len(self.labels))

    def _tok(self, strings, tokenizer):
        tl = [tokenizer(t, return_tensors='pt', padding='longest',
                        max_length=tokenizer.model_max_length, truncation=True)
              for t in strings]
        input_ids = [x.input_ids[0] for x in tl]
        lens = [x.input_ids.ne(tokenizer.pad_token_id).sum().item() for x in tl]
        return dict(input_ids=input_ids, input_ids_lens=lens)

    def __len__(self): return len(self.input_ids)
    def __getitem__(self, i): return dict(input_ids=self.input_ids[i], labels=self.labels[i])

@dataclass
class DataCollatorForSupervisedDataset(object):
    tokenizer: transformers.PreTrainedTokenizer
    def __call__(self, instances):
        input_ids, labels = tuple([inst[k] for inst in instances] for k in ('input_ids','labels'))
        input_ids = torch.nn.utils.rnn.pad_sequence(input_ids, batch_first=True,
                                                    padding_value=self.tokenizer.pad_token_id)
        labels = torch.nn.utils.rnn.pad_sequence(labels, batch_first=True, padding_value=IGNORE_INDEX)
        return dict(input_ids=input_ids, labels=labels,
                    attention_mask=input_ids.ne(self.tokenizer.pad_token_id))

train_dataset = SFT_dataset('data_kochatgpt/kochatgpt_1_SFT_clean.jsonl', tokenizer, verbose=True)
data_collator = DataCollatorForSupervisedDataset(tokenizer=tokenizer)

sample source: 아래는 작업을 설명하는 명령어입니다.
명령어에 따른 요청을 적절히 완료하는 응답을 작성하세요.

### In ...
loaded: 11800


In [11]:
# 학습. 빠르게 한 사이클 돌려보려고 1 epoch로 잡았다. 더 돌리고 싶으면 num_train_epochs만 올리면 된다.
training_args = TrainingArguments(
    output_dir='./sft_ckpt',
    num_train_epochs=1,
    per_device_train_batch_size=8,
    warmup_steps=5,
    logging_steps=100,
    save_steps=100000,          # 중간 저장은 생략(용량 절약)
    prediction_loss_only=True,
    fp16=torch.cuda.is_available(),
    report_to='none',
)
trainer = Trainer(model=model, args=training_args,
                  data_collator=data_collator, train_dataset=train_dataset)
trainer.train()

output_1_SFT = './output_1_SFT'
model.save_pretrained(output_1_SFT)
tokenizer.save_pretrained(output_1_SFT)
print('SFT 저장 완료:', output_1_SFT)

[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
100,3.286509
200,3.098351
300,3.058906
400,3.002528
500,2.979861
600,2.912906
700,2.864296
800,2.852439
900,2.861949
1000,2.837861


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

SFT 저장 완료: ./output_1_SFT


## Step 7. SFT 인퍼런스 + [기존 KoGPT2 vs SFT] 비교  
### (루브릭 3번)

SFT 모델로 같은 프롬프트에 답하게 하고, Step 5의 기존 KoGPT2 답과 나란히 놓고 비교한다. 정성 비교(같은 질문에 뭐라고 답하나)와 정량 비교(held-out eval셋에서 BLEU/ROUGE)를 같이 본다.

정량 지표는 한국어라서 그냥 계산하면 안 된다. BLEU/ROUGE 기본 토크나이저는 공백 기준(혹은 영숫자만)이라 한국어를 제대로 못 쪼갠다. 그래서 **KoGPT2 토크나이저로 subword 단위로 쪼갠 뒤** 지표를 계산하도록 맞췄다.

In [12]:
from transformers import pipeline
from rouge_score import rouge_scorer
import sacrebleu

sft_gen = pipeline('text-generation', model=output_1_SFT, tokenizer=tokenizer,
                   device=0 if torch.cuda.is_available() else -1)

gen_args = dict(max_new_tokens=128, repetition_penalty=2.0, no_repeat_ngram_size=4,
                do_sample=True, top_k=50, temperature=1.0, eos_token_id=tokenizer.eos_token_id)

def sft_answer(full_prompt, **kw):
    out = sft_gen(full_prompt, **{**gen_args, **kw})[0]['generated_text']
    return out.split('### Response(응답):')[-1].strip()

print('=== 기존 KoGPT2  vs  SFT 정성 비교 ===')
sft_outputs = {}
for raw, full in zip(eval_prompts_raw, eval_prompts):
    ans = sft_answer(full)
    sft_outputs[raw] = ans
    print('\nQ:', raw)
    print(' [기존]', base_outputs[raw][:150])
    print(' [SFT ]', ans[:150])

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'no_repeat_ngram_size', 'temperature', 'repetition_penalty', 'top_k', 'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


=== 기존 KoGPT2  vs  SFT 정성 비교 ===


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: 불고기용 고기 한우에요?
 [기존] food_delicious. 오늘은 제가 좋아하는 과자랑 함께 먹었어요.
요즘 날씨가 너무 추워서 그런지 더더욱 많이 먹고 싶었는데요.
오늘도 역시나 맛있는 과일들~!!
이렇게 맛있게 잘 먹는거 보니까 정말 기분이 좋네요
저는 이번에 처음 먹어보는 딸기잼과 딸기를 좋아
 [SFT ] 제가 알기로는, 불고기와 고기의 조합이 다른데, 이것은 양념이나 소스에 따라 다릅니다. 예를 들어 햄에 버터를 넣고 짠 뒤 다시 한번 구워먹거나, 고기를 잘게 부수어버린 후 적당한 두께로 썰어보시면 좋겠습니다만, 보통은 냉동 보관 시 온도가 섭씨 3도에서 7도로 떨어지


[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: 리처드 닉슨이 43대 부통령직을 수행한 년도는?
 [기존] beatmania_mygame.com/views/2648297.
이번주 토요일 오후 4시 30분부터 5시까지 서울 강남구 논현동 강남역 10길에서 열리는 ``2012 대한민국 게임대상 시상식 및 시상식: GAMESHOWN 2016-19 (Games of the Year
 [SFT ] 저는 더 많은 정보를 얻어야 합니다.
따라서, 리처드 닉슨이 46 대 대통령직 수행 기간 동안 41대를 역임했다고 알고 있습니다.


[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Q: 시카고 오헤어 국제공항은 어디에 있어
 [기존] jmtbc_kim (오늘도 좋은 하루 되세요)
이제 곧 여름휴가 시작!
아침부터 저녁까지 시원하게 보내주신다면! 오늘만큼 더위를 식혀줄 시원한 음료 한잔으로 즐거운 시간을 만들어보자구요
 
저는 이번에 처음 먹어보는 맥심 모카맛이라서 너무 좋았어요.
맥심은 정말 맛있었
 [SFT ] 저는 인공지능 언어 모델을 이용하여 시카고에 대한 정보를 알지 못합니다. 하지만 시카고 시에 위치한 미국항공공단(Chicago America Executive) 항공사는 2021년 기준으로 캐나다 퀘벡주에 위치하고 있기 때문에, 이 공항이 있는 곳이면 가까운 일리노이 

Q: 오늘 미세먼지 어때?
 [기존] selfie (반응:)
이제부터 시작!
미세 먼지가 많은 날엔, 마스크를 착용하고 외출해야 합니다.
마지막으로 중요한 것은 바로 '호흡기' 입니다.
평소보다 더 자주 숨을 쉬어야 하고, 특히나 호흡기가 약한 사람은 더욱 그렇습니다.
그렇다면 어떻게 해야 할까요?
우선 
 [SFT ] 죄송합니다, 저는 인공 지능 언어모델이며, 현재 날씨 정보나 관련된 정보를 가지고 있지 않습니다. 하지만 지역 및 상황에 따라 매우 상이할 수 있으니, 해당 지역의 기상 조건에 따라서 답변을 드릴 수밖에 없습니다! 가능하시다면 검색 엔진을 사용하여 주변 환경을 확인하여


In [13]:
# 정량 평가용 지표 함수 (한국어: KoGPT2 토크나이저로 쪼갠 뒤 계산)
class WSTokenizer:
    def tokenize(self, text): return text.split()
rouge = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False, tokenizer=WSTokenizer())

def ko_tok(text):
    return ' '.join(tokenizer.tokenize(text))

def eval_scores(pairs):
    """pairs: list of (reference, hypothesis) -> 평균 BLEU / ROUGE-1 / ROUGE-L"""
    bleus, r1s, rls = [], [], []
    for ref, hyp in pairs:
        if not hyp.strip(): hyp = ' '
        rt, ht = ko_tok(ref), ko_tok(hyp)
        bleus.append(sacrebleu.sentence_bleu(ht, [rt], tokenize='none').score)
        sc = rouge.score(rt, ht)
        r1s.append(sc['rouge1'].fmeasure); rls.append(sc['rougeL'].fmeasure)
    n = len(pairs)
    return dict(BLEU=sum(bleus)/n, ROUGE1=sum(r1s)/n, ROUGEL=sum(rls)/n)

In [14]:
# held-out eval셋에서 정량 비교 (기존 KoGPT2 vs SFT)
# eval은 200개 전부 돌리면 느리니 50개만 샘플링해서 비교한다. (더 정확히 보려면 늘리면 됨)
N_EVAL = 50
subset = eval_set[:N_EVAL]

base_pairs, sft_pairs = [], []
for ex in subset:
    full = PROMPT_DICT['prompt_no_input'].format_map({'prompt': ex['prompt']})
    ref  = ex['completion']
    b = base_generate(full).split('### Response(응답):')[-1].strip()
    s = sft_answer(full)
    base_pairs.append((ref, b))
    sft_pairs.append((ref, s))

base_score = eval_scores(base_pairs)
sft_score  = eval_scores(sft_pairs)
print('== 정량 비교 (held-out %d개, 기준: 원본 정답) ==' % N_EVAL)
print('기존 KoGPT2 :', {k: round(v,2) for k,v in base_score.items()})
print('SFT        :', {k: round(v,2) for k,v in sft_score.items()})

[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

== 정량 비교 (held-out 50개, 기준: 원본 정답) ==
기존 KoGPT2 : {'BLEU': 0.42, 'ROUGE1': 0.04, 'ROUGEL': 0.03}
SFT        : {'BLEU': 1.76, 'ROUGE1': 0.09, 'ROUGEL': 0.07}


기존 KoGPT2는 지시를 안 따르고 프롬프트 뒤를 아무렇게나 이어버리는 반면, SFT 모델은 질문에 대한 "응답" 형식으로 답을 만든다. 정량 지표(BLEU/ROUGE)도 SFT 쪽이 정답과 더 가까워지는 게 확인된다. **데이터 품질이 좋아진 상태에서 정답 부분만 학습시킨 효과**가 여기서 나온다.

## Step 8. Reward Model 학습

이제 RLHF의 두 번째 단계, "좋은 답 채점기"인 Reward Model을 만든다. RM 데이터는 하나의 프롬프트에 대해 completion 3개와 그 순위(ranking)가 있다. RM은 텍스트를 생성하는 게 아니라 **스칼라 점수(reward)를 출력**하고, "순위가 높은 답 > 낮은 답"이 되도록 학습한다(PairWiseLoss).

학습 형식이 (chosen, rejected) 쌍이라, ranking 3개를 서로 짝지어 (0vs1, 0vs2, 1vs2) 세 쌍으로 펼친다. 순위 숫자가 작을수록 좋은 답이라, 더 작은 쪽을 chosen으로 넣는다.

In [15]:
# GPTRM_custom: GPT2Model 위에 value_head(스칼라 출력)를 얹은 Reward Model
from typing import Optional
import torch.nn as nn
from transformers.models.gpt2.modeling_gpt2 import GPT2Model
from chatgpt.models.base import RewardModel

class GPTRM_custom(RewardModel):
    def __init__(self, pretrained=None, lora_rank=0, lora_train_bias='none', tokenizer=None):
        if pretrained is not None:
            model = GPT2Model.from_pretrained(pretrained)
            model.resize_token_embeddings(len(tokenizer))   # special token 추가분 반영
        else:
            from transformers.models.gpt2.configuration_gpt2 import GPT2Config
            model = GPT2Model(GPT2Config())
        value_head = nn.Linear(model.config.n_embd, 1)
        super().__init__(model, value_head, lora_rank, lora_train_bias)
        self.model = model
        self.pretrained = pretrained
    def save_pretrained(self, d):
        if self.pretrained is not None:
            self.model.save_pretrained(d)

In [16]:
from chatgpt.dataset import RewardDataset
from chatgpt.trainer import RewardModelTrainer
from chatgpt.trainer.strategies import NaiveStrategy
from torch.optim import Adam
from transformers import PreTrainedTokenizerFast

strategy = NaiveStrategy()

with strategy.model_init_context():
    # SFT와 동일하게 PreTrainedTokenizerFast로 고정 (AutoTokenizer는 GPT2Tokenizer로 폴백돼 한글이 깨진다)
    rm_tokenizer = PreTrainedTokenizerFast.from_pretrained(
        'skt/kogpt2-base-v2',
        bos_token='</s>', eos_token='</s>', unk_token='<unk>',
        pad_token='<pad>', mask_token='<mask>',
        padding_side='right', model_max_length=512)
    rm_model = GPTRM_custom(pretrained='skt/kogpt2-base-v2', tokenizer=rm_tokenizer).cuda()

# ranking → (chosen, rejected) 쌍 3개로 펼치기
rm_raw = load_json('data_kochatgpt/kochatgpt_2_RM.jsonl')
def pick(a_comp, a_rank, b_comp, b_rank):
    # rank 숫자가 작을수록 좋은 답 → 작은 쪽이 chosen
    return (a_comp, b_comp) if a_rank < b_rank else (b_comp, a_comp)

pairs = []
for t in rm_raw:
    r = t['ranking']
    for (i, j) in [(0,1),(0,2),(1,2)]:
        chosen, rejected = pick(t['completion_%d'%i], r[i], t['completion_%d'%j], r[j])
        pairs.append({'prompt': t['prompt'], 'chosen': chosen, 'rejected': rejected})

import random; random.seed(230319); random.shuffle(pairs)
print('RM 쌍 개수:', len(pairs))

# 코랩에서 빠르게 돌리려고 일부만 사용한다. 더 쓰고 싶으면 숫자를 늘리면 된다.
RM_TRAIN, RM_EVAL = 2000, 200
train_rm = RewardDataset(pairs[:RM_TRAIN], rm_tokenizer, max_length=256)
eval_rm  = RewardDataset(pairs[RM_TRAIN:RM_TRAIN+RM_EVAL], rm_tokenizer, max_length=256)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

[transformers] GPT2Model LOAD REPORT from: skt/kogpt2-base-v2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 
lm_head.weight                          | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RM 쌍 개수: 30660


100%|██████████| 200/200 [00:00<00:00, 1062.09it/s]


In [17]:
rm_optim = Adam(rm_model.parameters(), lr=5e-5)
rm_trainer = RewardModelTrainer(model=rm_model, strategy=strategy, optim=rm_optim,
                                train_dataset=train_rm, eval_dataset=eval_rm,
                                batch_size=4, max_epochs=1)
rm_trainer.fit(use_lora=0)

output_2_RM = './output_2_RM'
os.makedirs(output_2_RM, exist_ok=True)
rm_model.save_pretrained(output_2_RM)
print('RM 저장 완료:', output_2_RM)

Train epoch: 100%|██████████| 1/1 [03:48<00:00, 228.59s/it]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RM 저장 완료: ./output_2_RM


## Step 9. PPO (RLHF) 학습

마지막 단계. actor(=SFT 모델)가 답을 생성하면 reward model이 점수를 매기고, 그 점수를 최대화하는 방향으로 actor를 강화학습(PPO)한다. 동시에 initial model(고정된 SFT 복사본)에서 너무 멀어지지 않도록 KL 페널티가 걸린다 — 이게 없으면 점수만 좇다가 언어능력이 무너진다.

PPO는 SFT/RM보다 훨씬 무겁고 느려서, 여기서는 **동작 확인 규모**로 작게 돌린다(에피소드·프롬프트 수를 줄임). 규모를 키우려면 num_episodes와 프롬프트 개수를 늘리면 된다.

In [18]:
from copy import deepcopy
from chatgpt.models.gpt import GPTActor, GPTCritic
from chatgpt.models.base import RewardModel
from chatgpt.trainer import PPOTrainer

torch.cuda.empty_cache()
strategy = NaiveStrategy()

with strategy.model_init_context():
    # actor는 SFT 모델, critic은 RM 모델에서 출발
    actor  = GPTActor(pretrained=output_1_SFT).to(torch.cuda.current_device())
    critic = GPTCritic(pretrained=output_2_RM).to(torch.cuda.current_device())

    # SFT/RM과 동일하게 PreTrainedTokenizerFast로 고정
    ppo_tok = PreTrainedTokenizerFast.from_pretrained(
        'skt/kogpt2-base-v2',
        bos_token='</s>', eos_token='</s>', unk_token='<unk>',
        pad_token='<pad>', mask_token='<mask>',
        padding_side='right', model_max_length=512)

    initial_model = deepcopy(actor)                                  # KL 기준(고정)
    reward_model  = RewardModel(deepcopy(critic.model),
                                deepcopy(critic.value_head)).to(torch.cuda.current_device())

actor_optim  = Adam(actor.parameters(),  lr=5e-6)
critic_optim = Adam(critic.parameters(), lr=5e-6)

(actor, actor_optim), (critic, critic_optim), reward_model, initial_model = strategy.prepare(
    (actor, actor_optim), (critic, critic_optim), reward_model, initial_model)

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [19]:
# PPO에 쓸 프롬프트 (동작 확인용으로 일부만)
ppo_raw = load_json('data_kochatgpt/kochatgpt_3_PPO.jsonl')
list_prompt = [d['prompt'] for d in ppo_raw][:64]

def tokenize_fn(texts):
    batch = ppo_tok(texts, return_tensors='pt', max_length=96, padding=True, truncation=True)
    return {k: v.cuda() for k, v in batch.items()}

ppo_trainer = PPOTrainer(strategy, actor, critic, reward_model, initial_model,
                         actor_optim, critic_optim,
                         max_epochs=1, train_batch_size=8,
                         tokenizer=tokenize_fn, max_length=128,
                         do_sample=True, temperature=1.0, top_k=50,
                         pad_token_id=ppo_tok.pad_token_id, eos_token_id=ppo_tok.eos_token_id)

ppo_trainer.fit(list_prompt, num_episodes=1, max_timesteps=3, update_timesteps=3)

output_3_PPO = './output_3_PPO'
os.makedirs(output_3_PPO, exist_ok=True)
strategy.save_model(actor, os.path.join(output_3_PPO, 'actor.pt'), only_rank0=True)
print('PPO 저장 완료:', output_3_PPO)

Episode [1/1]: 100%|██████████| 3/3 [00:18<00:00,  6.30s/it]


PPO 저장 완료: ./output_3_PPO


## Step 10. PPO 인퍼런스 + [SFT vs PPO(RM 적용)] 비교  
### (루브릭 2번)

RLHF까지 거친 actor로 같은 프롬프트에 답하게 하고, SFT 단계 답과 비교한다. 정성 비교에 더해서, **Reward Model로 SFT 출력과 PPO 출력을 각각 채점**해서 정량적으로도 비교한다. RM이 준 점수가 올라갔다면, 강화학습이 "RM이 선호하는 방향"으로 actor를 움직였다는 뜻이다.

In [20]:
def ppo_generate(full_prompt):
    ids = ppo_tok.encode(full_prompt, return_tensors='pt').to(torch.cuda.current_device())
    out = actor.generate(ids, max_length=200, do_sample=True, top_k=50, top_p=0.95,
                         num_return_sequences=1)
    text = ppo_tok.batch_decode(out[0], skip_special_tokens=True)[0]
    return text.split('### Response(응답):')[-1].strip()

print('=== SFT  vs  PPO(RLHF) 정성 비교 ===')
ppo_outputs = {}
for raw, full in zip(eval_prompts_raw, eval_prompts):
    ans = ppo_generate(full)
    ppo_outputs[raw] = ans
    print('\nQ:', raw)
    print(' [SFT]', sft_outputs[raw][:150])
    print(' [PPO]', ans[:150])

=== SFT  vs  PPO(RLHF) 정성 비교 ===

Q: 불고기용 고기 한우에요?
 [SFT] 제가 알기로는, 불고기와 고기의 조합이 다른데, 이것은 양념이나 소스에 따라 다릅니다. 예를 들어 햄에 버터를 넣고 짠 뒤 다시 한번 구워먹거나, 고기를 잘게 부수어버린 후 적당한 두께로 썰어보시면 좋겠습니다만, 보통은 냉동 보관 시 온도가 섭씨 3도에서 7도로 떨어지
 [PPO] 제가 AI 언어 모델이므로, 고기용은 판매되는 지역에 따라 다르기 때문에 정확한 대답을 드릴 수 없습니다. 하지만 일반적으로 불고기의 경우 대형마트나 슈퍼마켓에서 주문하시는 것을 권장합니다. 급제한 고기나 된장, 볶음만두, 고칼로리 등 다양한 요리와 함께 즐길 수 있습

Q: 리처드 닉슨이 43대 부통령직을 수행한 년도는?
 [SFT] 저는 더 많은 정보를 얻어야 합니다.
따라서, 리처드 닉슨이 46 대 대통령직 수행 기간 동안 41대를 역임했다고 알고 있습니다.
 [PPO] 리처드 닉슨은 41대 부통령직을 수행한 것으로 알려져 있습니다.作作tion of anciential Conscience : Essayology of Phase Issue of the Years, of the Your Campsy and the Hellenolence, 

Q: 시카고 오헤어 국제공항은 어디에 있어
 [SFT] 저는 인공지능 언어 모델을 이용하여 시카고에 대한 정보를 알지 못합니다. 하지만 시카고 시에 위치한 미국항공공단(Chicago America Executive) 항공사는 2021년 기준으로 캐나다 퀘벡주에 위치하고 있기 때문에, 이 공항이 있는 곳이면 가까운 일리노이 
 [PPO] 시카고 오헤어 국제공항은 시카고 국제공항 주변에 위치해 있습니다.曉慧, 常世, 曹溪, 曹溪, 河, 常世, 曹溪, 曺溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪, 曹溪,

Q: 오늘 미세먼지 어때?
 [SFT] 

In [21]:
# RM 점수로 정량 비교: 같은 프롬프트에 대한 SFT 답 vs PPO 답을 RM이 채점
@torch.no_grad()
def rm_score(prompt, answer):
    text = prompt + answer + "</s>"
    enc = rm_tokenizer(text, max_length=256, padding='max_length',
                       truncation=True, return_tensors='pt')
    ids = enc['input_ids'].cuda(); mask = enc['attention_mask'].cuda()
    return rm_model(ids, attention_mask=mask).item()

rm_model.eval()
sft_rewards, ppo_rewards = [], []
for raw, full in zip(eval_prompts_raw, eval_prompts):
    sft_rewards.append(rm_score(full, sft_outputs[raw]))
    ppo_rewards.append(rm_score(full, ppo_outputs[raw]))

print('== RM 점수 비교 (높을수록 RM 선호) ==')
for raw, s, p in zip(eval_prompts_raw, sft_rewards, ppo_rewards):
    print('Q:%-28s SFT %.3f | PPO %.3f' % (raw[:26], s, p))
print('평균  SFT %.3f | PPO %.3f' % (sum(sft_rewards)/len(sft_rewards),
                                     sum(ppo_rewards)/len(ppo_rewards)))

== RM 점수 비교 (높을수록 RM 선호) ==
Q:불고기용 고기 한우에요?                SFT -0.649 | PPO -0.336
Q:리처드 닉슨이 43대 부통령직을 수행한 년도는?   SFT -0.800 | PPO -0.666
Q:시카고 오헤어 국제공항은 어디에 있어         SFT -0.535 | PPO -0.793
Q:오늘 미세먼지 어때?                  SFT -0.564 | PPO -0.501
평균  SFT -0.637 | PPO -0.574


여기서 결과를 있는 그대로 짚고 간다. RM 점수 평균은 PPO(-0.574)가 SFT(-0.637)보다 소폭 높았고, 4개 질문 중 3개에서 PPO가 우위였다. 동작 확인 규모(에피소드 1, 프롬프트 64개)치고는 RM이 선호하는 방향으로 actor가 미세하게 움직인 셈이다. 다만 점수 차가 작아서 이것만으로 RLHF 효과를 단정할 수는 없고, 중요한 건 **파이프라인이 실제로 SFT→RM→PPO로 이어지고, RM 점수라는 정량 축으로 두 모델을 비교할 수 있게 됐다**는 것이다. 규모를 키우면 이 축에서 차이가 더 벌어지는지 관찰할 수 있다.

## Step 11. 디코딩 기법 실험 + 정량 평가  
### (루브릭 1번)

같은 SFT 모델이라도 **디코딩 전략(어떻게 다음 토큰을 뽑나)**에 따라 출력이 꽤 달라진다. 모델 가중치는 그대로 두고 생성 방법만 바꿔가며 held-out eval셋에서 BLEU/ROUGE를 재서, 어떤 디코딩이 정답에 더 가까운 답을 만드는지 비교한다.

- **greedy**: 매 스텝 최고 확률 토큰만. 결정적이지만 단조로움.
- **beam search**: 여러 후보 경로를 동시에 탐색. 안정적이지만 무난한 답으로 수렴.
- **top-k**: 상위 k개 중에서 샘플링. 다양성↑.
- **top-p (nucleus)**: 누적확률 p까지에서 샘플링. 저확률 꼬리 차단.


In [22]:
# 디코딩 전략별 설정
decoding_configs = {
    'greedy'    : dict(do_sample=False, num_beams=1),
    'beam(4)'   : dict(do_sample=False, num_beams=4, no_repeat_ngram_size=4, early_stopping=True),
    'top_k(50)' : dict(do_sample=True,  top_k=50, temperature=1.0),
    'top_p(0.9)': dict(do_sample=True,  top_p=0.9, top_k=0, temperature=1.0),
}
common = dict(max_new_tokens=128, repetition_penalty=2.0, eos_token_id=tokenizer.eos_token_id)

N_DEC = 40                      # 속도상 40개로 비교 (늘리면 더 안정적)
dec_subset = eval_set[:N_DEC]

results = {}
for name, cfg in decoding_configs.items():
    pairs = []
    for ex in dec_subset:
        full = PROMPT_DICT['prompt_no_input'].format_map({'prompt': ex['prompt']})
        out = sft_gen(full, **{**common, **cfg})[0]['generated_text']
        hyp = out.split('### Response(응답):')[-1].strip()
        pairs.append((ex['completion'], hyp))
    results[name] = eval_scores(pairs)
    print('%-11s' % name, {k: round(v,2) for k,v in results[name].items()})

[transformers] Passing `generation_config` together with generation-related arguments=({'repetition_penalty', 'num_beams', 'do_sample', 'max_new_tokens', 'eos_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to 

greedy      {'BLEU': 2.47, 'ROUGE1': 0.12, 'ROUGEL': 0.1}


[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

beam(4)     {'BLEU': 6.15, 'ROUGE1': 0.2, 'ROUGEL': 0.18}


[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

top_k(50)   {'BLEU': 1.61, 'ROUGE1': 0.09, 'ROUGEL': 0.07}


[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=128) and `max_length`(=50) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/

top_p(0.9)  {'BLEU': 1.59, 'ROUGE1': 0.1, 'ROUGEL': 0.07}


In [23]:
# 표로 정리
import pandas as pd
df = pd.DataFrame(results).T[['BLEU','ROUGE1','ROUGEL']].round(2)
df.index.name = 'decoding'
print(df)
best = df['ROUGEL'].idxmax()
print('\nROUGE-L 기준 가장 좋은 디코딩:', best)

            BLEU  ROUGE1  ROUGEL
decoding                        
greedy      2.47    0.12    0.10
beam(4)     6.15    0.20    0.18
top_k(50)   1.61    0.09    0.07
top_p(0.9)  1.59    0.10    0.07

ROUGE-L 기준 가장 좋은 디코딩: beam(4)


디코딩만 바꿔도 지표가 눈에 띄게 움직인다. 보통 beam search가 정답과의 표면적 일치(BLEU/ROUGE)는 높게 나오는 경향이 있는데, 대신 답이 무난해지는 트레이드오프가 있다. 반대로 top-k/top-p는 다양하지만 정답과의 일치도는 흔들린다. **"지표가 높다 = 항상 좋다"가 아니라는 것**도 같이 봐야 해서, 아래 Step 12에서 실제 출력 예시도 같이 놓고 본다.

## Step 12. 종합 비교와 정성 분석

단계별(기존 KoGPT2 → SFT → PPO)과 디코딩별 결과를 한자리에 모은다.

In [24]:
print('==== 단계별 정량 비교 (held-out %d개) ====' % N_EVAL)
print('%-12s BLEU   ROUGE1  ROUGEL' % 'stage')
for name, sc in [('기존 KoGPT2', base_score), ('SFT', sft_score)]:
    print('%-12s %5.2f  %5.2f   %5.2f' % (name, sc['BLEU'], sc['ROUGE1'], sc['ROUGEL']))

print('\n==== 디코딩별 정량 비교 (SFT 모델) ====')
print(df)

print('\n==== 같은 질문, 단계별 답 (정성) ====')
for raw in eval_prompts_raw:
    print('\nQ:', raw)
    print(' 기존:', base_outputs[raw][:120])
    print(' SFT :', sft_outputs[raw][:120])
    print(' PPO :', ppo_outputs[raw][:120])

==== 단계별 정량 비교 (held-out 50개) ====
stage        BLEU   ROUGE1  ROUGEL
기존 KoGPT2     0.42   0.04    0.03
SFT           1.76   0.09    0.07

==== 디코딩별 정량 비교 (SFT 모델) ====
            BLEU  ROUGE1  ROUGEL
decoding                        
greedy      2.47    0.12    0.10
beam(4)     6.15    0.20    0.18
top_k(50)   1.61    0.09    0.07
top_p(0.9)  1.59    0.10    0.07

==== 같은 질문, 단계별 답 (정성) ====

Q: 불고기용 고기 한우에요?
 기존: food_delicious. 오늘은 제가 좋아하는 과자랑 함께 먹었어요.
요즘 날씨가 너무 추워서 그런지 더더욱 많이 먹고 싶었는데요.
오늘도 역시나 맛있는 과일들~!!
이렇게 맛있게 잘 먹는거 보니까 정말 기분이 
 SFT : 제가 알기로는, 불고기와 고기의 조합이 다른데, 이것은 양념이나 소스에 따라 다릅니다. 예를 들어 햄에 버터를 넣고 짠 뒤 다시 한번 구워먹거나, 고기를 잘게 부수어버린 후 적당한 두께로 썰어보시면 좋겠습니다만, 보
 PPO : 제가 AI 언어 모델이므로, 고기용은 판매되는 지역에 따라 다르기 때문에 정확한 대답을 드릴 수 없습니다. 하지만 일반적으로 불고기의 경우 대형마트나 슈퍼마켓에서 주문하시는 것을 권장합니다. 급제한 고기나 된장, 볶

Q: 리처드 닉슨이 43대 부통령직을 수행한 년도는?
 기존: beatmania_mygame.com/views/2648297.
이번주 토요일 오후 4시 30분부터 5시까지 서울 강남구 논현동 강남역 10길에서 열리는 ``2012 대한민국 게임대상 시상식 및 시상식: GAMESH
 SFT : 저는 더 많은 정보를 얻어야 합니다.
따라서, 리처드 닉슨이 46 대 대

### 정성 분석 요약
- **기존 KoGPT2 → SFT**: 가장 큰 변화가 여기서 일어났다. 기존 모델은 지시를 무시하고 프롬프트 뒤를 아무렇게나 이어붙이는데, SFT 후에는 "질문 → 응답" 형식으로 답이 잡히고 정량 지표(BLEU 0.42→1.76, ROUGE-L 0.03→0.07)도 같이 올라간다. 데이터 정제(따옴표·이스케이프·영어범벅 제거)가 이 학습의 바탕이 됐다. 다만 형식이 잡혔다고 내용까지 정확해진 건 아니어서, 닉슨 질문처럼 사실관계는 여전히 자주 틀린다 — 작은 모델의 한계이자, 형식 학습과 사실성은 별개 축이라는 걸 보여주는 예시다.
- **SFT → PPO**: RM 점수는 PPO가 소폭 우위였지만, 정성적으로 흥미로운 현상을 관찰했다. PPO 응답이 **앞부분은 응답 형식을 지키다가 후반부에 한자·영문 반복으로 붕괴**하는 경우가 있었다(시카고 질문의 "曹溪, 曹溪, ..." 반복이 대표적). 소규모 PPO에서 샘플링 생성(top-p 0.95, repetition penalty 없음)을 하다 보니, 강화학습으로 분포가 흔들린 영역에서 저품질 토큰이 한 번 뽑히면 그대로 반복 루프에 빠지는 것으로 보인다. RLHF가 "점수는 올리면서 언어능력을 망가뜨릴 수 있다"는 교과서적 위험을 축소판으로 직접 목격한 셈이다.
- **디코딩 실험**: 가중치를 안 건드리고 생성 방법만 바꿔도 지표가 크게 움직였다(ROUGE-L 기준 beam(4) 0.18 vs top-p 0.07, 2배 이상 차이). 특히 BLEU/ROUGE가 높은 디코딩과 실제로 읽기 좋은 답이 항상 일치하진 않는다는 것도 같이 봤다.

### 전체 실행 플로우

```
[Step 3-4] 데이터
  로드 → EDA(품질 신호 집계) → 정제(11,230개) → 증강(12,000개 복원) → train 11,800 / eval 200

[Step 5-7] SFT  (루브릭 3: 기존 KoGPT2 vs SFT)
  기존 KoGPT2 인퍼런스(베이스라인 고정) ─┐
  정제 데이터로 SFT 학습 ────────────────┴→ 정성 비교 + BLEU/ROUGE (0.42 → 1.76)

[Step 8-10] RLHF  (루브릭 2: SFT vs PPO)
  RM 데이터 ranking → (chosen, rejected) 쌍 → RM 학습(PairWiseLoss)
  SFT(actor) + RM(critic) → PPO 학습
  → SFT vs PPO 정성 비교 + RM 점수 정량 비교 (-0.637 vs -0.574)

[Step 11-12] 디코딩 실험  (루브릭 1: generation 기법 실험)
  같은 SFT 가중치 × greedy / beam(4) / top-k(50) / top-p(0.9)
  → BLEU/ROUGE 비교 (beam(4)이 ROUGE-L 0.18로 최고) → 종합 비교 → 회고
```

## 회고

이번 프로젝트에서 제일 크게 배운 건 **데이터 품질이 생성 성능의 바탕**이라는 걸 숫자로 확인한 거다. EDA를 해보니 SFT completion의 99% 이상이 stray 따옴표로 시작하고, 응답 안에 진짜 줄바꿈 대신 `\n` 글자가 그대로 박혀 있거나 한글 없이 영어로만 답한 샘플도 꽤 있었다. 이걸 그냥 두고 학습했으면 모델이 그 노이즈까지 흉내냈을 거다. 정제 규칙을 EDA에서 나온 실제 숫자에 근거해서 잡은 게 이번 작업의 핵심이었다.

겪었던 문제도 몇 개 있었다. 우선 원본 노드가 colossalai를 쓰는데, 옛날 torch 조합이 코랩에서 설치가 안 됐다. single GPU만 쓸 거라 `strategies/__init__.py`에서 colossalai 전략 import만 걷어내고 naive 전략만 남기니 패키지가 정상적으로 import됐다 — 코드를 처음부터 다시 읽으면서 어디서 colossalai에 걸리는지 import 흐름을 추적한 게 도움이 됐다. 또 하나는 정량 평가였다. 처음에 BLEU/ROUGE를 그냥 쟀더니 ROUGE가 계속 0이 나와서 한참 헤맸는데, rouge 기본 토크나이저가 영숫자만 남기고 한글을 통째로 날려버리는 게 원인이었다. KoGPT2 토크나이저로 subword를 쪼갠 뒤 커스텀 토크나이저로 넘기니 정상적으로 점수가 잡혔다. "돌아가긴 하는데 값이 이상한" 종류의 버그라, 지표가 0이라고 그냥 넘기지 않고 왜 0인지 원인을 판 게 맞았다.

이번에 제일 인상 깊었던 버그는 토크나이저였다. 학습은 멀쩡히 돌고 loss도 내려가는데 생성 결과가 전부 `�`로 깨져 나왔고, BLEU/ROUGE도 바닥이었다. 에러가 한 줄도 안 나는 상태라 처음엔 학습 실패를 의심했는데, 기존 KoGPT2 베이스라인(Step 5)은 한국어가 멀쩡했다는 게 단서였다. 두 단계의 차이를 하나씩 좁혀보니 로드 방식이 달랐다 — 베이스라인은 `PreTrainedTokenizerFast`로, 나머지는 `AutoTokenizer`로 로드했는데, 최신 transformers에서 AutoTokenizer가 kogpt2를 byte-level BPE(`GPT2Tokenizer`)로 잡아버려서 vocab과 인코딩/디코딩이 통째로 어긋난 거였다. 모델·데이터·학습 코드가 다 정상이어도 **토크나이저 짝 하나가 어긋나면 파이프라인 전체가 조용히 무너진다**는 걸 몸으로 확인했고, "돌아가는데 결과만 이상하면 제일 먼저 텍스트↔토큰 변환 경계를 의심하라"는 체크리스트가 하나 생겼다.

트랙 관점에서 스스로 정한 선도 지켰다고 생각한다. 이번 프로젝트는 하이퍼파라미터를 붙잡고 점수를 극한까지 끌어올리는 게 목적이 아니라, **정제와 디코딩이 모델 출력을 어떻게 바꾸는지 관찰**하는 게 목적이었다. 그래서 augmentation도 정교하게 튜닝하지 않고 instruction만 다양화하는 단순하고 안전한 방법으로만 했고, 대신 각 단계(기존→SFT→PPO)와 디코딩별로 출력이 어떻게 달라지는지를 정량·정성 양쪽으로 비교하는 데 시간을 더 썼다.

다음에 시도해볼 것: PPO는 동작 확인 규모라 RM 점수가 소폭 오르는 것까지만 확인했는데, 대신 후반부 생성이 한자 반복으로 붕괴하는 현상을 관찰했다. 다음엔 에피소드와 프롬프트 수를 늘려 RM 점수가 유의미하게 올라가는지 보되, PPO 모델의 생성에도 repetition penalty와 no_repeat_ngram을 걸어 붕괴가 샘플링 문제인지 정책 자체의 손상인지 분리해보고 싶다. KL 페널티 계수를 조절하면서 "점수는 오르는데 언어가 망가지는" 지점이 어디서 오는지 보는 것도 흥미로울 것 같다. 또 정량 지표(BLEU/ROUGE)가 높은 답과 사람이 보기에 좋은 답이 갈리는 지점을 더 파보면, 나중에 에이전트 출력 품질을 평가할 때 "지표만 믿으면 안 되는" 감각으로 이어질 것 같다.